# Plan B: WeatherDistill, knowledge distillation for adverse-weather detection

Run the cells from top to bottom. Read `plans/02-weather-distill.md` first. Before you start:

1. Use **Runtime > Change runtime type** and pick a GPU (for example T4).
2. Nothing to upload: section 3 downloads the four DAWN zips (139 MB) from Mendeley Data and checks the SHA-256 values in the `dawn` entry of `datasets/manifest.json`. DAWN is for educational use only: keep every copy private.
3. Edit the **Configuration** cell: the mode, the fold, the seed and the number of epochs.

Layout after the download (folder names may differ in case; images and annotation files may sit in sub-folders):

```
dawn_raw/
  Fog/   Rain/   Snow/   Sand/
```

## 1. Install and check the GPU

In [ ]:
# Clone the project (or update it when the runtime is reused), install the pinned requirements and show the GPU
import os
import subprocess
import sys
from pathlib import Path
# Address of the team repository
repositoryUrl = "https://github.com/keithku321/object-detection"
# Folder on the Colab machine that holds the clone
repositoryFolder = Path("/content/objectDetection")
# A private repository needs a GitHub token stored as the Colab Secret GITHUB_TOKEN (key icon in the left bar); never paste a token into a cell
try:
    from google.colab import userdata
    githubToken = userdata.get("GITHUB_TOKEN")
except Exception:
    githubToken = None
cloneUrl = repositoryUrl.replace("https://", f"https://{githubToken}@") if githubToken else repositoryUrl
gitCommand = ["git", "-C", str(repositoryFolder), "pull", "--quiet"] if repositoryFolder.exists() else ["git", "clone", "--quiet", cloneUrl, str(repositoryFolder)]
gitResult = subprocess.run(gitCommand, capture_output=True, text=True)
# Print git's message with the token removed, so the token can never end up in a saved notebook
if gitResult.returncode != 0: print(gitResult.stderr.replace(githubToken or "GITHUB_TOKEN", "***"))
os.chdir(repositoryFolder)
if str(repositoryFolder) not in sys.path: sys.path.insert(0, str(repositoryFolder))
# Colab already has PyTorch; requirements.txt adds the pinned Ultralytics and the task libraries without touching it
!pip install -q -r requirements.txt
# Show the attached GPU; if this prints an error then change the runtime type to a GPU
!nvidia-smi
from common.environment import printEnvironment
printEnvironment()

## 2. Configuration

In [ ]:
# Name of this task; it names the results file and the run folders
taskName = "weather"
# Dataset entry in datasets/manifest.json
datasetName = "dawn"
# Name of the folder that the download is unzipped into; it must match extractFolder of the dawn entry in datasets/manifest.json
rawFolderName = "dawn_raw"
# What to train in section 7: lambda0, feat, cls, box, full, full-fg or full-both-heads (see weather/distillTrainer.py)
distillMode = "full"
# Held-out fold (0 to 4); lambda and temperature are tuned on fold 0 only
foldIndex = 0
# Seed; exploration uses 0, the final runs use 0, 1 and 2 on every fold
seed = 0
# Number of full passes over the training data; DAWN is small, so nano and medium models train quickly
trainingEpochs = 100
# Image size in pixels; student, teacher and distilled student must use the same value
imageSize = 640
# Images per batch
batchSize = 16
# Drive folder for results, so that weights survive when Colab resets the runtime
driveResultsFolder = Path("/content/drive/MyDrive/yoloProjectResults")
# Local folder with the raw download
rawRootPath = Path("/content/datasets") / rawFolderName
# Local folder that will hold the YOLO-format dataset
datasetRootPath = Path("/content/datasets/dawn")

## 3. Mount Google Drive and unpack the dataset

The four zips go straight to the local disk of the Colab machine (training directly from Drive is slow) and each is checked against the SHA-256 that Mendeley Data publishes, which is recorded in `datasets/manifest.json`.

In [ ]:
from google.colab import drive
# Mount Google Drive; a pop-up will ask for permission. Drive holds results and checkpoints only, never the working copy of the dataset
drive.mount("/content/drive")
driveResultsFolder.mkdir(parents=True, exist_ok=True)
# Bring back results rows of earlier sessions: a fresh clone only contains the rows that were committed
from common.results import syncResultFiles
print(syncResultFiles(taskName, driveResultsFolder / "results"), "results rows so far")

from common.datasets import fetchDataset
# Download the four weather zips from Mendeley Data, verify them and unzip them into /content/datasets/dawn_raw
fetchDataset(datasetName, "/content/datasets")

## 4. Helper functions

The logic lives in `common/` and `weather/` so it can be tested on a PC; this cell only imports it and adds two display helpers.

In [ ]:
from IPython.display import display
from PIL import Image
# File endings that count as images
imageSuffixes = {".jpg", ".jpeg", ".png"}

# Display the first few images from a folder inside the notebook
def showSavedImages(folderPath, maximumCount=4):
    imageFiles = [file for file in sorted(Path(folderPath).iterdir()) if file.suffix.lower() in imageSuffixes]
    for imageFile in imageFiles[:maximumCount]:
        display(Image.open(imageFile))

# Save the results rows to Drive and print the summary table (mean ± std over seeds)
def saveAndShowResults():
    from common.results import syncResultFiles
    from common.summariseResults import writeSummary
    syncResultFiles(taskName, driveResultsFolder / "results")
    print(writeSummary() or "No results rows yet.")

from common.datasets import findLabelProblems
from weather.prepareDawn import assignFolds, buildFoldYamls, convertDawn, countClassIds, dawnSplitsFolder, getDawnSettings, writeFoldLists
from weather.train import getTeacherWeights, runFold

## 5. Create dataset.yaml and sanity-check the data

DAWN is converted to YOLO format with one folder per weather type. The YOLO files of the download number their classes like COCO counting from 1, so the numbers are renumbered by class name (`sourceYoloClassIds` in the manifest). The five folds are stratified by weather, and near-duplicate images are kept in the same fold. The fold lists in `common/splits/dawn/` are committed: once they exist, everybody must use the same lists, so they are only written when they are missing.

Read any warnings carefully. **Check the class order**: compare the boxes-per-class counts below with a few images before you trust any number.

In [ ]:
weatherTypes, classNames = getDawnSettings()
# Convert the raw download to <dataset>/images/<weather> and <dataset>/labels/<weather>
imagesByWeather = convertDawn(rawRootPath, datasetRootPath, weatherTypes, classNames)
print({weatherType: len(imagePaths) for weatherType, imagePaths in imagesByWeather.items()})
print("Boxes per class id:", countClassIds(datasetRootPath), "for classes", classNames)
for weatherType in weatherTypes:
    for problem in findLabelProblems(datasetRootPath, weatherType, len(classNames)): print("  WARNING:", problem)
# Write the fold lists only when the repository does not have them yet, then commit common/splits/dawn/
if not (dawnSplitsFolder / "fold0.txt").exists():
    writeFoldLists(assignFolds(imagesByWeather), datasetRootPath)
    print("New fold lists written to", dawnSplitsFolder, "- commit them so the whole team uses the same folds")
yamlByFold = buildFoldYamls(datasetRootPath, classNames)
print(Path(yamlByFold[foldIndex]).read_text())

## 6. Baseline

Two reference runs on the chosen fold: the plain student `yolo26n` (experiment W0, what distillation has to beat) and the teacher `yolo26m` (W1, the upper reference). The teacher of a fold is trained only on that fold's training images. Finished runs are skipped, so this cell is cheap to re-run.

In [ ]:
# Plain student and teacher on this fold; each logs an overall row and one row per weather type
studentWeightsPath = runFold("student", foldIndex, seed, trainingEpochs, datasetRootPath, driveResultsFolder, imageSize, batchSize)
teacherWeightsPath = runFold("teacher", foldIndex, 0, trainingEpochs, datasetRootPath, driveResultsFolder, imageSize, batchSize)
saveAndShowResults()

## 7. Train

Distil the teacher into the student. Results are written straight to Drive so the weights survive a disconnect. The loss columns `dis_loss`, `cls_kd_loss` and `box_kd_loss` are the feature, class and box distillation terms.

Run mode `lambda0` once before any real run: with every distillation weight at zero it must reproduce the plain student within seed noise.

In [ ]:
# Train the distilled student of this fold and log an overall row and one row per weather type
distilledWeightsPath = runFold(distillMode, foldIndex, seed, trainingEpochs, datasetRootPath, driveResultsFolder, imageSize, batchSize)
saveAndShowResults()

### If Colab disconnected during training

Run the earlier setup cells (1 to 5) again, then run the cell below. It is the same call as the training cell: every run has a fixed folder on Drive, so training continues from the last saved epoch, and finished runs are skipped.

In [ ]:
# Continue the interrupted run from its last checkpoint on Drive (same call as above; finished runs are skipped)
# distilledWeightsPath = runFold(distillMode, foldIndex, seed, trainingEpochs, datasetRootPath, driveResultsFolder, imageSize, batchSize)
# saveAndShowResults()

## 8. Evaluate the trained model

Accuracy rows are already logged. This cell checks the claim that distillation costs nothing at inference: the distilled student must have exactly as many parameters as the plain student, and the same speed.

In [ ]:
from common.benchmark import benchmarkModel
from weather.train import checkStudentSize
# The distilled student must be a plain yolo26n: nothing of the teacher or the projector may remain in the weights
assert checkStudentSize(distilledWeightsPath, studentWeightsPath), "the distilled student is not the same size as the plain student"
for modelLabel, weightsPath in (("student", studentWeightsPath), ("distilled", distilledWeightsPath), ("teacher", teacherWeightsPath)):
    print(modelLabel, benchmarkModel(weightsPath))
saveAndShowResults()

## 9. Look at real predictions

Scores hide failures, so look at the images, especially dense fog and heavy snow.

In [ ]:
from ultralytics import YOLO
from common.training import endToEndArguments
from weather.prepareDawn import heldOutImagesByWeather
# Predict on a few held-out images of every weather type with the distilled student and save annotated copies
distilledModel = YOLO(str(distilledWeightsPath))
for weatherType, imagePaths in heldOutImagesByWeather(datasetRootPath, foldIndex, weatherTypes).items():
    distilledModel.predict(source=[str(path) for path in imagePaths[:2]], save=True, conf=0.4, project="/content/runs/trained", name="predict", exist_ok=True, **endToEndArguments)
showSavedImages("/content/runs/trained/predict", maximumCount=8)

## 10. Evaluate by weather type

This is the cell that turns the project into a finding: does distillation help more in some weather than in others? The table has one row per variant and weather type (`student@fog`, `distill-full@fog`, ...), averaged over the folds and seeds run so far.

In [ ]:
from common.results import readResultRows, resultsFolder
# Print mAP50 by weather type for every variant logged so far
weatherRows = [row for row in readResultRows(resultsFolder / "weather.csv") if "@" in row["variant"]]
for row in sorted(weatherRows, key=lambda row: (row["variant"].split("@")[1], row["variant"], row["seed"])):
    print(f"{row['variant']:<28} seed {row['seed']}  {row['notes'].split(';')[0]:<8} mAP50 = {float(row['map50']):.3f}  ({row['images']} images)")

## 11. Export and download (optional)

Download the weights for the demo app (`weights/` folder of the repository). Export only if you need another format.
The results rows now sit in `results/weather.csv` inside the clone and in `yoloProjectResults/results/` on Drive. Rows are part of the repository: download the CSV (or copy it from Drive) and commit it on your branch, without weights. Strip the outputs of this notebook before committing it (**Edit > Clear all outputs**).

In [ ]:
from google.colab import files
# Download the final weights to your own computer; the file name follows <task>-<variant>-seed<k>.pt
files.download(str(distilledWeightsPath))
# Export to ONNX; this is also the starting point of the INT8 experiment in weather/quantise.py
exportedModelPath = distilledModel.export(format="onnx", imgsz=imageSize)
print(exportedModelPath)
saveAndShowResults()